# Notebook 05 — Brand & Listing Quality Analysis

**Amazon Market Intelligence**  
**Questions answered:** Q5 (Who are my competitors and how do they win?), Q6 (How does my product compare?)  
**Tool modes:** Competitive Positioning, Health Check  
**Gold tables:** `gold_brand_dynamics` (~496), `gold_listing_quality` (248), `gold_bestseller_analysis` (248)

---

## 0 — Setup

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'amazon_intelligence.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)

CHARTS_DIR = 'charts/05_brand_listing_quality'
os.makedirs(CHARTS_DIR, exist_ok=True)

TEMPLATE = 'plotly_white'

def save_chart(fig, name):
    fig.write_html(f'{CHARTS_DIR}/{name}.html')
    try:
        fig.write_image(f'{CHARTS_DIR}/{name}.png', width=1200, height=700, scale=2)
    except Exception:
        pass

print(f'Connected to: {DB_PATH}')
print(f'Charts → {CHARTS_DIR}/')

Connected to: c:\Users\thinkpad\Desktop\amazon-market-intelligence\data\amazon_intelligence.duckdb
Charts → charts/05_brand_listing_quality/


## 1 — Schema Discovery

In [2]:
for table in ['gold_brand_dynamics', 'gold_listing_quality', 'gold_bestseller_analysis']:
    print(f'\n=== {table} ===')
    print(con.sql(f'DESCRIBE {table}').df().to_string())
    print(f'Rows: {con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0]:,}')


=== gold_brand_dynamics ===
                column_name column_type null   key default extra
0               subcategory     VARCHAR  YES  None    None  None
1              brand_status     VARCHAR  YES  None    None  None
2             product_count      BIGINT  YES  None    None  None
3                 avg_price      DOUBLE  YES  None    None  None
4                avg_rating      DOUBLE  YES  None    None  None
5               avg_reviews      DOUBLE  YES  None    None  None
6          total_units_sold     HUGEINT  YES  None    None  None
7             total_revenue      DOUBLE  YES  None    None  None
8   avg_revenue_per_product      DOUBLE  YES  None    None  None
9                pct_active      DOUBLE  YES  None    None  None
10         pct_best_sellers      DOUBLE  YES  None    None  None
11         avg_discount_pct      DOUBLE  YES  None    None  None
Rows: 491

=== gold_listing_quality ===
                    column_name column_type null   key default extra
0                

## 2 — Load Gold Tables

In [3]:
df_brand = con.sql('SELECT * FROM gold_brand_dynamics').df()
df_listing = con.sql('SELECT * FROM gold_listing_quality').df()
df_bs = con.sql('SELECT * FROM gold_bestseller_analysis').df()

print(f'Brand dynamics: {df_brand.shape}')
print(f'Listing quality: {df_listing.shape}')
print(f'Bestseller analysis: {df_bs.shape}')

df_brand.head(3)

Brand dynamics: (491, 12)
Listing quality: (248, 21)
Bestseller analysis: (248, 16)


,subcategory,brand_status,product_count,avg_price,avg_rating,avg_reviews,total_units_sold,total_revenue,avg_revenue_per_product,pct_active,pct_best_sellers,avg_discount_pct
0,Abrasive & Finishing Products,Unbranded,6628,24.79,3.23,0.4,105550.0,1678527.0,253.25,12.6,0.2,17.5
1,Abrasive & Finishing Products,Branded,2018,23.07,4.33,0.0,54750.0,795024.5,393.97,22.6,0.1,17.9
2,Accessories & Supplies,Unbranded,3101,40.23,4.40,0.0,553850.0,20001956.0,6450.16,14.8,3.0,23.4


In [4]:
df_listing.head(3)

,subcategory,product_count,avg_title_length,median_title_length,pct_with_features,pct_with_description,pct_with_brand,pct_with_store,avg_rev_with_features,avg_rev_without_features,...,avg_rev_without_description,avg_rev_with_brand,avg_rev_without_brand,avg_rev_short_title,avg_rev_medium_title,avg_rev_long_title,avg_rev_very_long_title,avg_completeness_score,avg_rev_high_completeness,avg_rev_low_completeness
0,Girls' Clothing,28619,69.6,71.0,0.8,0.5,0.0,0.8,734.82,232.03,...,232.90,1190.50,235.81,275.83,203.61,289.37,292.27,0.02,837.30,232.05
1,Boys' Clothing,24660,63.3,61.0,1.2,0.7,0.0,1.2,191.41,249.08,...,248.36,343.56,248.35,269.05,238.22,232.52,93.11,0.03,261.35,249.09
2,Toys & Games,20846,153.4,165.0,33.2,16.2,10.2,33.5,7830.41,5832.12,...,6268.63,6238.55,6524.11,5148.08,5171.98,7237.15,6461.63,0.93,7562.57,5844.26


In [5]:
df_bs.head(3)

,subcategory,total_products,bestseller_count,pct_bestsellers,avg_price_bestseller,avg_price_non_bestseller,avg_rating_bestseller,avg_rating_non_bestseller,avg_reviews_bestseller,avg_reviews_non_bestseller,avg_rev_bestseller,avg_rev_non_bestseller,avg_sales_bestseller,avg_sales_non_bestseller,pct_revenue_from_bestsellers,bestseller_revenue_multiplier
0,Sports & Fitness,6662,483.0,7.25,26.23,26.78,4.50,4.45,0.0,0.0,28301.76,12786.68,1228.4,557.5,14.7,2.2
1,Industrial & Scientific,4433,404.0,9.11,19.97,19.91,4.56,4.55,0.0,0.0,63190.51,25978.46,3197.0,1440.0,19.6,2.4
2,Automotive Replacement Parts,8289,368.0,4.44,26.21,23.93,4.48,4.48,0.0,0.0,11139.40,4099.45,592.3,200.9,11.2,2.7


---

## 3 — Brand Dynamics

Finding #26: Brand multiplier ranges from 207× (Sony PSP) to ~1×.  
Finding #18: Brand advantage flips in Beauty.  
The question: **is having a brand a competitive moat, or does it depend entirely on category?**

In [6]:
BRAND_CAT_COL = 'subcategory'
BRAND_STATUS_COL = 'brand_status'
BRAND_REVENUE = 'total_revenue'
BRAND_PRODUCTS = 'product_count'
BRAND_AVG_RATING = 'avg_rating'
BRAND_AVG_SALES = 'avg_units_sold'

print(df_brand.columns.tolist())
print(f'\nBrand status values: {df_brand[BRAND_STATUS_COL].unique()}')

['subcategory', 'brand_status', 'product_count', 'avg_price', 'avg_rating', 'avg_reviews', 'total_units_sold', 'total_revenue', 'avg_revenue_per_product', 'pct_active', 'pct_best_sellers', 'avg_discount_pct']

Brand status values: ['Unbranded' 'Branded']


### 3.1 — Branded vs Unbranded: Overall

In [7]:
brand_totals = (
    df_brand.groupby(BRAND_STATUS_COL)
    .agg(
        total_rev=(BRAND_REVENUE, 'sum'),
        total_products=(BRAND_PRODUCTS, 'sum')
    )
    .reset_index()
)
brand_totals['rev_per_product'] = brand_totals['total_rev'] / brand_totals['total_products'].replace(0, np.nan)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Total Revenue', 'Revenue per Product'],
    horizontal_spacing=0.15
)

fig.add_trace(go.Bar(
    x=brand_totals[BRAND_STATUS_COL], y=brand_totals['total_rev'],
    marker_color=['#2196F3', '#FF9800'][:len(brand_totals)],
    text=[f'${v/1e6:.0f}M' for v in brand_totals['total_rev']],
    textposition='outside', showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=brand_totals[BRAND_STATUS_COL], y=brand_totals['rev_per_product'],
    marker_color=['#2196F3', '#FF9800'][:len(brand_totals)],
    text=[f'${v:,.0f}' for v in brand_totals['rev_per_product']],
    textposition='outside', showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Branded vs Unbranded — Overall Performance',
    template=TEMPLATE, height=450,
    margin=dict(t=80)
)
save_chart(fig, '01_branded_vs_unbranded_overall')
fig.show()

### 3.2 — Brand Multiplier by Category

The brand advantage ratio: branded revenue per product ÷ unbranded revenue per product.  
This is the chart that proves brand value is category-dependent.

In [8]:
df_brand['rev_per_product'] = df_brand[BRAND_REVENUE] / df_brand[BRAND_PRODUCTS].replace(0, np.nan)

brand_pivot = df_brand.pivot_table(
    index=BRAND_CAT_COL, columns=BRAND_STATUS_COL,
    values='rev_per_product', aggfunc='first'
)

print(f'Brand status columns in pivot: {brand_pivot.columns.tolist()}')

Brand status columns in pivot: ['Branded', 'Unbranded']


In [9]:
BRANDED_LABEL = 'Branded'      
UNBRANDED_LABEL = 'Unbranded'  

brand_pivot['brand_multiplier'] = (
    brand_pivot[BRANDED_LABEL] / brand_pivot[UNBRANDED_LABEL].replace(0, np.nan)
)
brand_pivot = brand_pivot.dropna(subset=['brand_multiplier'])

top_brand = brand_pivot.nlargest(15, 'brand_multiplier')
bottom_brand = brand_pivot.nsmallest(15, 'brand_multiplier')
extreme = pd.concat([bottom_brand, top_brand]).sort_values('brand_multiplier')
extreme = extreme[extreme['brand_multiplier'] > 0]

colors = ['#F44336' if x < 1 else '#2196F3' for x in extreme['brand_multiplier']]

fig = go.Figure(go.Bar(
    x=np.log2(extreme['brand_multiplier']),
    y=extreme.index,
    orientation='h',
    marker_color=colors,
    text=[f'{v:.1f}×' for v in extreme['brand_multiplier']],
    textposition='outside'
))
fig.add_vline(x=0, line_color='black', line_width=2)
fig.update_layout(
    title='Brand Multiplier by Category — Where Branding Matters (and Where It Doesn\'t)',
    xaxis_title='Branded ÷ Unbranded Revenue/Product (log scale)',
    template=TEMPLATE, height=700
)
save_chart(fig, '02_brand_multiplier')
fig.show()

### 3.3 — Brand Presence Across Categories

What % of products are branded in each category? Some categories are brand-dominated, others are commodity markets.

In [10]:
brand_share = df_brand.pivot_table(
    index=BRAND_CAT_COL, columns=BRAND_STATUS_COL,
    values=BRAND_PRODUCTS, aggfunc='first'
).fillna(0)

brand_share['total'] = brand_share.sum(axis=1)
brand_share['branded_pct'] = (brand_share[BRANDED_LABEL] / brand_share['total'] * 100).round(1)
brand_share = brand_share.sort_values('branded_pct', ascending=True)

most_branded = brand_share.nlargest(20, 'branded_pct')
least_branded = brand_share.nsmallest(20, 'branded_pct')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Most Branded Categories', 'Least Branded (Commodity Markets)'],
    horizontal_spacing=0.4
)

fig.add_trace(go.Bar(
    x=most_branded['branded_pct'], y=most_branded.index,
    orientation='h', marker_color='#2196F3',
    text=[f'{v:.0f}%' for v in most_branded['branded_pct']],
    textposition='inside', showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=least_branded['branded_pct'], y=least_branded.index,
    orientation='h', marker_color='#FF9800',
    text=[f'{v:.1f}%' for v in least_branded['branded_pct']],
    textposition='inside', showlegend=False
), row=1, col=2)

fig.update_layout(title='Brand Penetration by Category', template=TEMPLATE, height=700, width=1600)
save_chart(fig, '03_brand_presence')
fig.show()

---

## 4 — Listing Quality Impact

Finding #38: Listing completeness multiplies revenue up to 27×.  
Finding #10: Listing richness threshold at 3 features.  

Does having features, descriptions, brands, and store info actually matter?

### 4.1 — Revenue by Listing Completeness

In [11]:
elements = {
    'Features': ('avg_rev_with_features', 'avg_rev_without_features'),
    'Description': ('avg_rev_with_description', 'avg_rev_without_description'),
    'Brand': ('avg_rev_with_brand', 'avg_rev_without_brand'),
    'High Completeness': ('avg_rev_high_completeness', 'avg_rev_low_completeness')
}

multipliers = []
for label, (with_col, without_col) in elements.items():
    ratio = (df_listing[with_col] / df_listing[without_col].replace(0, np.nan)).median()
    multipliers.append({'element': label, 'median_multiplier': ratio})

df_mult = pd.DataFrame(multipliers).sort_values('median_multiplier', ascending=True)

fig = px.bar(
    df_mult,
    x='median_multiplier', y='element',
    orientation='h',
    title='Listing Element Impact — Median Revenue Multiplier (With ÷ Without)',
    labels={'median_multiplier': 'Median Multiplier (×)', 'element': ''},
    template=TEMPLATE,
    color_discrete_sequence=['#4CAF50']
)
fig.update_traces(texttemplate='%{x:.1f}×', textposition='outside')
fig.add_vline(x=1, line_color='gray', line_dash='dash')
fig.update_layout(height=350)
save_chart(fig, '04_listing_element_multipliers')
fig.show()

### 4.2 — Listing Completeness Multiplier by Category

How much more does a complete listing earn vs an incomplete one, per category?

In [12]:
df_listing['completeness_multiplier'] = (
    df_listing['avg_rev_high_completeness'] / 
    df_listing['avg_rev_low_completeness'].replace(0, np.nan)
)

top20_completeness = df_listing.dropna(subset=['completeness_multiplier']).nlargest(20, 'completeness_multiplier')

fig = px.bar(
    top20_completeness,
    x='completeness_multiplier', y='subcategory',
    orientation='h',
    title='Top 20 Categories: High vs Low Completeness Revenue Multiplier',
    labels={'completeness_multiplier': 'High ÷ Low Completeness (×)', 'subcategory': ''},
    template=TEMPLATE,
    color_discrete_sequence=['#4CAF50']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=650)
fig.update_traces(texttemplate='%{x:.1f}×', textposition='outside')
save_chart(fig, '05_completeness_multiplier')
fig.show()

### 4.3 — Listing Quality Components

Which listing elements matter most? Features, description, brand, or store presence?

In [13]:
df_sub = con.sql('SELECT * FROM gold_subcategory_landscape').df()

df_sub['rev_per_product'] = df_sub['total_revenue'] / df_sub['product_count'].replace(0, np.nan)

quality_cols = ['pct_with_brand', 'pct_with_features', 'pct_with_description', 'pct_with_store']
correlations = df_sub[quality_cols + ['rev_per_product']].corr()['rev_per_product'].drop('rev_per_product')
correlations = correlations.sort_values()

fig = px.bar(
    x=correlations.values,
    y=correlations.index.str.replace('pct_with_', '').str.title(),
    orientation='h',
    title='Which Listing Elements Correlate Most with Revenue?',
    labels={'x': 'Correlation with Revenue per Product', 'y': ''},
    template=TEMPLATE,
    color_discrete_sequence=['#9C27B0']
)
fig.update_traces(texttemplate='%{x:.3f}', textposition='outside')
fig.update_layout(height=350)
save_chart(fig, '06_listing_components_correlation')
fig.show()

---

## 5 — Best Seller Badge Analysis

Finding #39: Best Seller badge = 132× revenue in Travel Accessories.  
Finding #6: 1,738 Best Seller products with zero sales.  

The badge is coveted — but is it actually predictive, or is it just a lagging indicator?

In [14]:
BS_CAT_COL = 'subcategory'              
BS_REVENUE_MULT = 'revenue_multiplier'  
BS_COUNT = 'bestseller_count'           
BS_PCT = 'pct_bestsellers'

print(df_bs.columns.tolist())

['subcategory', 'total_products', 'bestseller_count', 'pct_bestsellers', 'avg_price_bestseller', 'avg_price_non_bestseller', 'avg_rating_bestseller', 'avg_rating_non_bestseller', 'avg_reviews_bestseller', 'avg_reviews_non_bestseller', 'avg_rev_bestseller', 'avg_rev_non_bestseller', 'avg_sales_bestseller', 'avg_sales_non_bestseller', 'pct_revenue_from_bestsellers', 'bestseller_revenue_multiplier']


### 5.1 — Best Seller Badge Distribution

In [15]:
fig = px.histogram(
    df_bs,
    x=BS_PCT,
    nbins=25,
    title='Distribution of Best Seller Badge Rates Across Categories',
    labels={BS_PCT: '% of Products with Best Seller Badge'},
    template=TEMPLATE,
    color_discrete_sequence=['#FF9800']
)
fig.add_vline(x=df_bs[BS_PCT].median(), line_dash='dash', line_color='red',
              annotation_text=f'Median: {df_bs[BS_PCT].median():.1f}%')
fig.update_layout(height=400)
save_chart(fig, '07_bestseller_distribution')
fig.show()

### 5.2 — Best Seller Revenue Multiplier

How much more do Best Seller products earn compared to non-Best Sellers in the same category?

In [16]:
BS_CAT_COL = 'subcategory'
BS_PCT = 'pct_bestsellers'

df_bs['badge_multiplier'] = df_bs['bestseller_revenue_multiplier']

top_badge = df_bs.dropna(subset=['badge_multiplier']).nlargest(20, 'badge_multiplier')

fig = px.bar(
    top_badge,
    x='badge_multiplier', y=BS_CAT_COL,
    orientation='h',
    title='Top 20 Categories: Best Seller Badge Revenue Multiplier',
    labels={'badge_multiplier': 'Best Seller ÷ Non-Best Seller (×)', BS_CAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#FF9800']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=650)
fig.update_traces(texttemplate='%{x:.1f}×', textposition='outside')
save_chart(fig, '08_bestseller_multiplier')
fig.show()

In [17]:
df_bs['badge_multiplier'] = df_bs['bestseller_revenue_multiplier']

top_badge = df_bs.dropna(subset=['badge_multiplier']).nlargest(20, 'badge_multiplier')

fig = px.bar(
    top_badge,
    x='badge_multiplier', y='subcategory',
    orientation='h',
    title='Top 20 Categories: Best Seller Badge Revenue Multiplier',
    labels={'badge_multiplier': 'Best Seller ÷ Non-Best Seller (×)', 'subcategory': ''},
    template=TEMPLATE,
    color_discrete_sequence=['#FF9800']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=650)
fig.update_traces(texttemplate='%{x:.1f}×', textposition='outside')
save_chart(fig, '08_bestseller_multiplier')
fig.show()

### 5.3 — Best Seller ≠ Best Product

Finding #6: 1,738 Best Seller products have zero sales. The badge is a lagging indicator.

In [18]:
if 'pct_active' in df_bs.columns:
    fig = px.scatter(
        df_bs,
        x=BS_PCT, y='pct_active',
        hover_name=BS_CAT_COL,
        title='Best Seller Badge Rate vs Activity Rate — Badge ≠ Alive',
        labels={BS_PCT: '% Best Sellers', 'pct_active': '% Active Products'},
        template=TEMPLATE,
        color_discrete_sequence=['#E91E63']
    )
    fig.update_layout(height=500)
    save_chart(fig, '09_badge_vs_activity')
    fig.show()
else:

    df_bs_merged = df_bs.merge(
        df_sub[[BRAND_CAT_COL, 'pct_active']],
        left_on=BS_CAT_COL, right_on=BRAND_CAT_COL, how='left'
    )
    fig = px.scatter(
        df_bs_merged,
        x=BS_PCT, y='pct_active',
        hover_name=BS_CAT_COL,
        title='Best Seller Badge Rate vs Activity Rate — Badge ≠ Alive',
        labels={BS_PCT: '% Best Sellers', 'pct_active': '% Active Products'},
        template=TEMPLATE,
        color_discrete_sequence=['#E91E63']
    )
    fig.update_layout(height=500)
    save_chart(fig, '09_badge_vs_activity')
    fig.show()

---

## 6 — Combined View: What Predicts Success?

Bring brand, listing quality, and bestseller signals together. Which matters most?

In [19]:
signals = df_sub[[
    'rev_per_product', 'pct_with_brand', 'pct_with_features',
    'pct_with_description', 'pct_with_store', 'pct_best_sellers',
    'avg_rating', 'avg_reviews', 'pct_active'
]].copy()

signals.columns = [
    'Revenue/Product', 'Brand %', 'Features %',
    'Description %', 'Store %', 'Best Seller %',
    'Avg Rating', 'Avg Reviews', 'Activity %'
]

corr = signals.corr()

fig = px.imshow(
    corr,
    title='Success Signal Correlation Matrix',
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    text_auto='.2f',
    aspect='auto'
)
fig.update_layout(height=600)
save_chart(fig, '10_success_correlation_matrix')
fig.show()

---

## 7 — Key Findings

In [20]:
print('=' * 60)
print('BRAND & LISTING QUALITY — KEY FINDINGS')
print('=' * 60)

print(f'\n1. BRAND IMPACT (OVERALL):')
for _, row in brand_totals.iterrows():
    print(f'   {row[BRAND_STATUS_COL]}: ${row["total_rev"]/1e6:.0f}M total, ${row["rev_per_product"]:,.0f}/product')

if 'brand_multiplier' in brand_pivot.columns:
    max_mult = brand_pivot['brand_multiplier'].max()
    max_cat = brand_pivot['brand_multiplier'].idxmax()
    min_mult = brand_pivot['brand_multiplier'].min()
    min_cat = brand_pivot['brand_multiplier'].idxmin()
    print(f'\n2. BRAND MULTIPLIER RANGE:')
    print(f'   Highest: {max_cat} ({max_mult:.1f}×)')
    print(f'   Lowest: {min_cat} ({min_mult:.1f}×)')

print(f'\n3. LISTING ELEMENT MULTIPLIERS (median across categories):')
for _, row in df_mult.iterrows():
    print(f'   {row["element"]}: {row["median_multiplier"]:.1f}×')

print(f'\n4. LISTING COMPONENTS (correlation with revenue):')
for comp, corr_val in correlations.items():
    label = comp.replace('pct_with_', '').title()
    print(f'   {label}: {corr_val:+.3f}')

print(f'\n5. BEST SELLER BADGE:')
print(f'   Median badge rate: {df_bs[BS_PCT].median():.1f}%')
print(f'   Categories with >5% badges: {(df_bs[BS_PCT] > 5).sum()}')
top1_badge = df_bs.nlargest(1, 'badge_multiplier').iloc[0]
print(f'   Highest multiplier: {top1_badge["subcategory"]} ({top1_badge["badge_multiplier"]:.1f}×)')

print('\n' + '=' * 60)

BRAND & LISTING QUALITY — KEY FINDINGS

1. BRAND IMPACT (OVERALL):
   Branded: $1004M total, $3,634/product
   Unbranded: $3647M total, $3,171/product

2. BRAND MULTIPLIER RANGE:
   Highest: Sony PSP Games, Consoles & Accessories (206.1×)
   Lowest: Boys' Shoes (0.0×)

3. LISTING ELEMENT MULTIPLIERS (median across categories):
   Description: 0.7×
   High Completeness: 0.9×
   Features: 0.9×
   Brand: 0.9×

4. LISTING COMPONENTS (correlation with revenue):
   Description: -0.001
   Features: +0.077
   Store: +0.087
   Brand: +0.200

5. BEST SELLER BADGE:
   Median badge rate: 0.3%
   Categories with >5% badges: 8
   Highest multiplier: Wii Games, Consoles & Accessories (997.2×)



In [21]:
con.close()
print('Done. DuckDB connection closed.')
print(f'Charts saved to: {CHARTS_DIR}/')

Done. DuckDB connection closed.
Charts saved to: charts/05_brand_listing_quality/
